# 01 · Data loading + checks
**Owner:** Senanayake · **Branch:** `data/01-loading-checks`

Read all 3 batches from the MATR/Severson `.mat` structs, apply the canonical cell removals + batch1↔batch2 continuation merge, and run the data-quality checks (missing / duplicate / invalid / cycle-order / ranges).


In [1]:
# --- Colab setup (run first) ---
import os
REPO = '/content/ev-battery-rul-prediction'   # absolute path -> safe to re-run
if not os.path.exists(REPO):
    !git clone https://github.com/Chathumi-Senanayake/ev-battery-rul-prediction.git {REPO}
%cd {REPO}
!git pull -q
!pip install -q -r requirements.txt
import sys
if REPO not in sys.path: sys.path.insert(0, REPO)

# Try Google Drive (data persists across sessions). If auth fails, fall back
# to local Colab storage so you are never blocked.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/ev_battery_data'
except Exception as e:
    print('Drive mount failed -> using local storage:', e)
    DATA_DIR = '/content/ev_battery_data'
os.makedirs(DATA_DIR, exist_ok=True)
print('DATA_DIR =', DATA_DIR)


Cloning into '/content/ev-battery-rul-prediction'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 56 (delta 37), reused 41 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 23.41 KiB | 4.68 MiB/s, done.
Resolving deltas: 100% (37/37), done.
/content/ev-battery-rul-prediction
Mounted at /content/drive
DATA_DIR = /content/drive/MyDrive/ev_battery_data


### Download (run once)
All 3 batch URLs are built into `src/data_loading.py`. Files go to `DATA_DIR`. On local storage they are wiped if the runtime restarts, so re-run this cell after any restart.


In [2]:
from src.data_loading import DOWNLOAD, MAT_FILENAMES
for b in ['batch1','batch2','batch3']:
    out = os.path.join(DATA_DIR, MAT_FILENAMES[b])
    if os.path.exists(out): print('have', MAT_FILENAMES[b]); continue
    !wget -q -O '{out}' '{DOWNLOAD[b]}' && echo 'downloaded {b}'


downloaded batch1
downloaded batch2
downloaded batch3


### Load + merge
`load_batch(..., lean=True)` (default) keeps summary + curve arrays only, so all 3 batches fit in a standard Colab runtime. If you hit a RAM crash, use **Runtime → Change runtime type → High-RAM**.


In [3]:
import pickle, gc
from src.data_loading import load_batch, merge_batches, batch_counts
b1 = load_batch(os.path.join(DATA_DIR, MAT_FILENAMES['batch1']), 'b1')
b2 = load_batch(os.path.join(DATA_DIR, MAT_FILENAMES['batch2']), 'b2')
b3 = load_batch(os.path.join(DATA_DIR, MAT_FILENAMES['batch3']), 'b3')
print('raw cells:', len(b1), len(b2), len(b3))
print('expected:', batch_counts(b1, b2, b3))
bat_dict = merge_batches(b1, b2, b3)
print('merged:', len(bat_dict), '(expect 124)')
pickle.dump(bat_dict, open(os.path.join(DATA_DIR,'bat_dict.pkl'),'wb'))
del b1, b2, b3; gc.collect()


raw cells: 46 48 46
expected: {'batch1': 41, 'batch2': 43, 'batch3': 40, 'total': 124}
merged: 124 (expect 124)


0

### Quality checks


In [4]:
from src.preprocessing import build_summary_table, quality_report, print_report
df = build_summary_table(bat_dict)
print_report(quality_report(df))
df.head()


rows: 100501   cells: 124
duplicate (cell,cycle): 0
bad cycle order cells : none
IR<=0: 62   QD<=0: 41
out of range: {'QD': 4, 'QC': 29, 'IR': 0, 'Tavg': 41, 'Tmin': 41, 'Tmax': 41, 'chargetime': 14}
missing (non-zero): none


,cell,batch,cycle_life,charge_policy,IR,QC,QD,Tavg,Tmin,Tmax,chargetime,cycle
0,b1c0,b1,1852.0,3.6C(80%)-3.6C,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
1,b1c0,b1,1852.0,3.6C(80%)-3.6C,0.016742,1.071042,1.070689,31.875011,29.566130,35.652016,13.341250,2.0
2,b1c0,b1,1852.0,3.6C(80%)-3.6C,0.016724,1.071674,1.071900,31.931490,29.604385,35.692978,13.425777,3.0
3,b1c0,b1,1852.0,3.6C(80%)-3.6C,0.016681,1.072304,1.072510,31.932603,29.744202,35.680588,13.425167,4.0
4,b1c0,b1,1852.0,3.6C(80%)-3.6C,0.016662,1.072970,1.073174,31.959322,29.644709,35.728691,13.341442,5.0


In [6]:
print('DATA_DIR =', DATA_DIR)
import os
print(os.listdir(DATA_DIR))

DATA_DIR = /content/drive/MyDrive/ev_battery_data
['2017-05-12_batchdata_updated_struct_errorcorrect.mat', '2017-06-30_batchdata_updated_struct_errorcorrect.mat', '2018-04-12_batchdata_updated_struct_errorcorrect.mat', 'bat_dict.pkl']
